# Setup

In [1]:
# Load necessary packages. 
import os
import numpy as np
import pandas as pd
import pyBigWig
import matplotlib.pyplot as plt
import subprocess
import importlib
import utils as utils
importlib.reload(utils)

<module 'utils' from '/tscc/projects/ps-yeolab3/kflanagan/plip_plop/V2_feature_vectors/preprocess_prototyping/utils.py'>

In [2]:
# Load up initial comparisons. 
comparison_table = pd.read_csv("../select_comparisons/encode_initial_30_comparisons.tsv", sep="\t")

# Choose an arbitrary comparison to observe. 
comparison_id = 0

comparison = comparison_table.iloc[comparison_id]

experiment_A = comparison["experiment_A"]
experiment_B = comparison["experiment_B"]

print("Category:", comparison["category"])
print("Experiment A:", experiment_A)
print("Experiment B:", experiment_B)

Category: similar
Experiment A: FUS_HepG2_ENCSR464OSH
Experiment B: TAF15_HepG2_ENCSR841EQA


# Build window to observe

In [3]:
# Remake the regions to observe. 
regions = utils.load_regions([comparison["window_A"], comparison["window_B"]], buffer=100, block_size=300)

# Calculate library sizes. 
library_sizes = {"IP_A_1": utils.get_mapped_reads(comparison["bam_1_A"]), "IP_A_2": utils.get_mapped_reads(comparison["bam_2_A"]),
                 "IN_A_1": utils.get_mapped_reads(comparison["bam_IN_1_A"]), "IP_B_1": utils.get_mapped_reads(comparison["bam_1_B"]),
                 "IP_B_2": utils.get_mapped_reads(comparison["bam_2_B"]), "IN_B_1": utils.get_mapped_reads(comparison["bam_IN_1_B"])}

# Organize library sizes. 
ip_A_library_sizes = [library_sizes["IP_A_1"], library_sizes["IP_A_2"]]
in_A_library_sizes = [library_sizes["IN_A_1"]]
ip_B_library_sizes = [library_sizes["IP_B_1"], library_sizes["IP_B_2"]]
in_B_library_sizes = [library_sizes["IN_B_1"]]

In [5]:
# Extract bigwig signal for all groups. 
ip_A = utils.extract_bigwig_signal(utils.get_track_paths(comparison, "plus", "IP", "A"), utils.get_track_paths(comparison, "minus", "IP", "A"), regions)
in_A = utils.extract_bigwig_signal(utils.get_track_paths(comparison, "plus", "IN", "A"), utils.get_track_paths(comparison, "minus", "IN", "A"), regions)
ip_B = utils.extract_bigwig_signal(utils.get_track_paths(comparison, "plus", "IP", "B"), utils.get_track_paths(comparison, "minus", "IP", "B"), regions)
in_B = utils.extract_bigwig_signal(utils.get_track_paths(comparison, "plus", "IN", "B"), utils.get_track_paths(comparison, "minus", "IN", "B"), regions)

In [6]:
# Confirm all extracted datasets contain the same number of windows.

if len({len(ip_A), len(in_A), len(ip_B), len(in_B)}) != 1:
    raise ValueError("Signal tracks do not contain the same number of genomic blocks.")

# Initialize list to hold window data.

window_data = []

# Run through all datasets and assemble windows.

for a_ip, a_in, b_ip, b_in in zip(ip_A, in_A, ip_B, in_B):
    coordinates = [
        (x["chr"], x["start"], x["end"], x["strand"])
        for x in [a_ip, a_in, b_ip, b_in]
    ]

    if len(set(coordinates)) != 1:
        raise ValueError("Signal tracks do not contain matching genomic blocks.")

    window_data.append({
        "chr": a_ip["chr"],
        "start": a_ip["start"],
        "end": a_ip["end"],
        "strand": a_ip["strand"],
        "region_id": a_ip["region_id"],
        "block_number": a_ip["block_number"],

        "ip_replicates_A": a_ip["replicate_signals"],
        "in_replicates_A": a_in["replicate_signals"],
        "ip_replicates_B": b_ip["replicate_signals"],
        "in_replicates_B": b_in["replicate_signals"]
    })

print("Windows assembled:", len(window_data))

Windows assembled: 17981


In [7]:
window_data[1]

{'chr': 'chr1',
 'start': 630973,
 'end': 631273,
 'strand': '+',
 'region_id': 1,
 'block_number': 0,
 'ip_replicates_A': array([[  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
           0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
           0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
           0.,   0.,   0.,   0.,   0.,   0.,   2.,   2.,   2.,   2.,   2.,
           2.,   2.,   2.,   4.,   4.,   4.,  10.,  10.,  11.,  12.,  14.,
          16.,  17.,  18.,  18.,  18.,  18.,  18.,  18.,  18.,  18.,  18.,
          18.,  18.,  18.,  18.,  19.,  19.,  19.,  19.,  19.,  20.,  20.,
          21.,  21.,  21.,  21.,  22.,  24.,  26.,  26.,  27.,  28.,  30.,
          32.,  33.,  32.,  31.,  29.,  29.,  27.,  27.,  27.,  27.,  26.,
          28.,  30.,  30.,  30.,  32.,  33.,  34.,  37.,  38.,  39.,  40.,
          41.,  42.,  42.,  44.,  45.,  44.,  49.,  49.,  49.,  49.,  49.,
          53.,  55.,  56.,  57.,  61.,  64.,  65.,  6